# Open Darts - fine-tune the dart detector on your own photos (Kaggle, free GPU)

Continues your trained `best.pt` on **your own labelled photos** mixed with a sample of the public data (so it does not forget what it knows). About 1.5 hours on a T4 instead of 6.

**Before you start (on your PC)**
```
python training/from_captures.py Screenshots/Training --out my_photos
python training/from_captures.py Screenshots/Holdout --out my_holdout
python training/pack_for_kaggle.py my_photos my_photos.zip
python training/pack_for_kaggle.py my_holdout my_holdout.zip
```
Upload both zips to Google Drive (Share > Anyone with the link) and paste the links in cell 1, or attach them as a Kaggle dataset (+ Add Input).
`my_holdout` is **never trained on**: it is only used to watch the progress, so the final check on it with `eval_real.py` stays honest.

Setup and running are the same as in `kaggle_training.ipynb`: Accelerator = GPU T4, Internet = On, then **Save Version > Save & Run All (Commit)**.

In [ ]:
# 1. Settings. Paste a Drive share link (or just its file ID) between the quotes.
DEEPDARTS_DRIVE = ""   # cropped_images.zip (public DeepDarts pictures)
YOLO_TAR_DRIVE = ""    # yolo_darts.tar (the prepared public dataset)
WEIGHTS_DRIVE = ""     # best.pt of the model to continue from (required)
MY_PHOTOS_DRIVE = ""   # my_photos.zip   (your labelled photos, made by from_captures.py + pack_for_kaggle.py)
MY_HOLDOUT_DRIVE = ""  # my_holdout.zip  (the photos kept out of training, only used to watch progress)
EPOCHS = 8
MY_REPEAT = 3          # how many times each of your pictures is in an epoch
PUBLIC_SAMPLE = 4000   # public pictures mixed in, so the model keeps what it learned from them
ALLOW_DARTSCRIBE_ONLY = False

In [ ]:
# 2. Machine check, helpers, the project code.
import os, subprocess, sys, glob, shutil, zipfile

import torch
assert torch.cuda.is_available(), "No GPU: Session options > Accelerator > GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0), "| GPUs:", torch.cuda.device_count())

WORK = "/kaggle/working"
TEMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"   # scratch space: not part of the notebook output
REPO = f"{WORK}/open-darts"
SCRIPTS = f"{REPO}/training"
DATASET = f"{TEMP}/dataset"
YOLO_DIR = f"{TEMP}/yolo_darts"
RUNS = f"{WORK}/dartnet_runs"


def sh(command, check=True):
    """Runs a shell command and shows its output as it comes (progress bars included)."""
    print("$", command, flush=True)
    process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    fd = process.stdout.fileno()
    while True:
        chunk = os.read(fd, 4096)
        if not chunk:
            break
        sys.stdout.write(chunk.decode("utf-8", "replace"))
        sys.stdout.flush()
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f"command failed with exit code {code}: {command}")
    return code


def drive_get(link, output):
    """Downloads a file shared on Google Drive ("Anyone with the link"); [link] is the share link or the bare file ID."""
    import gdown
    link = link.strip()
    print("downloading from Drive to", output, flush=True)
    done = gdown.download(link, output, quiet=False, fuzzy=True) if link.startswith("http") else gdown.download(id=link, output=output, quiet=False)
    if not done or not os.path.exists(output):
        raise RuntimeError(
            "Drive download failed. Is the file shared as 'Anyone with the link'? If Drive says too many people downloaded it, "
            "make a copy of the file in your own Drive and share the copy."
        )
    return output


sh("pip -q install ultralytics pandas pyyaml gdown")
if os.path.isdir(f"{REPO}/.git"):
    sh(f"git -C {REPO} pull --ff-only", check=False)
else:
    sh(f"git clone https://github.com/Yaswanth1826/open-darts.git {REPO}")
assert os.path.exists(f"{SCRIPTS}/train.py"), "the project did not clone: is Internet switched on in Session options?"

In [ ]:
# 3. DeepDarts pictures (the 800 px version of cropped_images.zip) and labels.pkl. Kaggle cannot log in to IEEE for you,
# so the zip has to come from your Drive (DEEPDARTS_DRIVE) or from a Kaggle dataset you attached.
def find_800(root):
    for path, dirs, _ in os.walk(root):
        if os.path.basename(path) == "800" and any(d[:3] in ("d1_", "d2_") for d in dirs):
            return path
    return None


def find_files(root, name):
    return [p for p in glob.glob(f"{root}/**/{name}", recursive=True)]


pictures = find_800(DATASET) or find_800("/kaggle/input")
if pictures is None:
    archive = next(iter(find_files("/kaggle/input", "cropped_images*.zip")), None)
    if archive is None and DEEPDARTS_DRIVE:
        archive = drive_get(DEEPDARTS_DRIVE, f"{TEMP}/cropped_images.zip")
    if archive is not None:
        print("unzipping", archive)
        with zipfile.ZipFile(archive) as z:
            z.extractall(DATASET)
        pictures = find_800(DATASET)

if pictures is None and not ALLOW_DARTSCRIBE_ONLY:
    raise SystemExit(
        "The DeepDarts pictures were not found, so nothing was trained (this saves you hours of GPU time on a weaker model).\\n"
        "Fix: put the Drive share link of cropped_images.zip in DEEPDARTS_DRIVE (cell 1; in Drive right-click the zip > Share > "
        "Anyone with the link > Copy link), or attach the zip as a Kaggle dataset (+ Add Input). "
        "Or set ALLOW_DARTSCRIBE_ONLY = True."
    )

os.makedirs(f"{DATASET}/cropped_images", exist_ok=True)
if pictures is not None and os.path.abspath(pictures) != os.path.abspath(f"{DATASET}/cropped_images/800"):
    if os.path.lexists(f"{DATASET}/cropped_images/800"):
        os.remove(f"{DATASET}/cropped_images/800")
    os.symlink(pictures, f"{DATASET}/cropped_images/800")
print("DeepDarts pictures:", pictures or "NONE (dartscribe only)")
sh(f"wget -q -O {DATASET}/labels.pkl https://github.com/wmcnally/deep-darts/raw/master/dataset/labels.pkl")
assert os.path.getsize(f"{DATASET}/labels.pkl") > 1000, "labels.pkl did not download"

In [ ]:
# 4. One YOLO dataset: DeepDarts (flat) + side-view copies + the dartscribe side photos. Taken ready-made from yolo_darts.tar if you have it
# (a minute or two), otherwise built here (about an hour).
def adopt_tar(tar):
    """Unpacks a yolo_darts.tar made on Colab and repairs it: its pictures point at /content/dataset and data.yaml at /content/yolo_darts."""
    shutil.rmtree(YOLO_DIR, ignore_errors=True)
    sh(f"tar -xf {tar} -C {TEMP}")
    if not os.path.isdir(YOLO_DIR):
        return False
    broken = 0
    for split in ("train", "val", "test"):
        folder = f"{YOLO_DIR}/images/{split}"
        for entry in os.scandir(folder):
            if entry.is_symlink() and not os.path.exists(entry.path):
                target = os.readlink(entry.path)
                at = target.find("cropped_images/")
                new = f"{DATASET}/{target[at:]}" if at >= 0 else None
                os.remove(entry.path)
                if new and os.path.exists(new):
                    os.symlink(new, entry.path)
                else:
                    broken += 1
                    if broken <= 3:
                        print("picture missing:", target)
    with open(f"{YOLO_DIR}/data.yaml", "w") as f:
        f.write(f"path: {YOLO_DIR}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: dart\n  1: cal1\n  2: cal2\n  3: cal3\n  4: cal4\n")
    print("pictures that could not be linked:", broken)
    return broken == 0


if not os.path.exists(f"{YOLO_DIR}/.ready"):
    tar = next(iter(find_files("/kaggle/input", "yolo_darts.tar")), None)
    if tar is None and YOLO_TAR_DRIVE:
        tar = drive_get(YOLO_TAR_DRIVE, f"{TEMP}/yolo_darts.tar")
    if tar is not None:
        if adopt_tar(tar):
            open(f"{YOLO_DIR}/.ready", "w").write("ok")
        else:
            print("the tar does not fit these pictures, building the dataset from scratch instead")
            shutil.rmtree(YOLO_DIR, ignore_errors=True)
    if not os.path.exists(f"{YOLO_DIR}/.ready"):
        sh(f"python {SCRIPTS}/prepare_deepdarts.py --dataset-dir {DATASET} --out {YOLO_DIR} --oblique 2")
        sh(f"python {SCRIPTS}/prepare_dartscribe.py --download {TEMP}/dartscribe")
        sh(f"python {SCRIPTS}/prepare_dartscribe.py --dataset-dir {TEMP}/dartscribe --out {YOLO_DIR}")
        open(f"{YOLO_DIR}/.ready", "w").write("ok")
else:
    print("the dataset is already prepared, skipping")
for split in ("train", "val", "test"):
    print(split, len(os.listdir(f"{YOLO_DIR}/images/{split}")), "pictures")

In [ ]:
# 5. The training mix: your photos (repeated) + a random sample of the public pictures; your hold-out photos are only for watching progress.
import random

def fetch_zip(name, drive, dest):
    archive = next(iter(find_files("/kaggle/input", name)), None)
    if archive is None and drive:
        archive = drive_get(drive, f"{TEMP}/{name}")
    if archive is None:
        raise SystemExit(f"{name} not found: paste its Drive link in cell 1 or attach it as a Kaggle dataset.")
    shutil.rmtree(dest, ignore_errors=True)
    with zipfile.ZipFile(archive) as z:
        z.extractall(dest)
    assert os.path.isdir(f"{dest}/images") and os.path.isdir(f"{dest}/labels"), f"{name} must contain images/ and labels/ (make it with pack_for_kaggle.py)"
    return dest

MINE = fetch_zip("my_photos.zip", MY_PHOTOS_DRIVE, f"{TEMP}/my_photos")
HOLD = fetch_zip("my_holdout.zip", MY_HOLDOUT_DRIVE, f"{TEMP}/my_holdout")
MIX = f"{TEMP}/yolo_mix"
shutil.rmtree(MIX, ignore_errors=True)
for split in ("train", "val"):
    os.makedirs(f"{MIX}/images/{split}")
    os.makedirs(f"{MIX}/labels/{split}")

rng = random.Random(0)
names = sorted(os.listdir(f"{YOLO_DIR}/images/train"))
rng.shuffle(names)
public = 0
for name in names:
    if public >= PUBLIC_SAMPLE:
        break
    stem = os.path.splitext(name)[0]
    src, label = f"{YOLO_DIR}/images/train/{name}", f"{YOLO_DIR}/labels/train/{stem}.txt"
    if os.path.exists(src) and os.path.exists(label):
        os.symlink(os.path.realpath(src), f"{MIX}/images/train/{name}")
        shutil.copy(label, f"{MIX}/labels/train/{stem}.txt")
        public += 1

mine = 0
for name in sorted(os.listdir(f"{MINE}/images")):
    stem = os.path.splitext(name)[0]
    for k in range(MY_REPEAT):
        os.symlink(f"{MINE}/images/{name}", f"{MIX}/images/train/mine{k}_{name}")
        shutil.copy(f"{MINE}/labels/{stem}.txt", f"{MIX}/labels/train/mine{k}_{stem}.txt")
        mine += 1
for name in sorted(os.listdir(f"{HOLD}/images")):
    stem = os.path.splitext(name)[0]
    shutil.copy(f"{HOLD}/images/{name}", f"{MIX}/images/val/{name}")
    shutil.copy(f"{HOLD}/labels/{stem}.txt", f"{MIX}/labels/val/{stem}.txt")
with open(f"{MIX}/data.yaml", "w") as f:
    f.write(f"path: {MIX}\ntrain: images/train\nval: images/val\ntest: images/val\nnames:\n  0: dart\n  1: cal1\n  2: cal2\n  3: cal3\n  4: cal4\n")
print(f"train: {public} public + {mine} of your pictures ({MY_REPEAT}x) | val: {len(os.listdir(f'{MIX}/images/val'))} hold-out pictures")

In [ ]:
# 6. Train: continue from your best.pt with a gentle learning rate. Weights are saved after every epoch under /kaggle/working/dartnet_runs.
prior = next(iter(find_files("/kaggle/input", "best.pt")), None)
if prior is None and WEIGHTS_DRIVE:
    prior = drive_get(WEIGHTS_DRIVE, f"{TEMP}/earlier_best.pt")
assert prior, "best.pt is required: paste its Drive link in WEIGHTS_DRIVE (cell 1) or attach it as a Kaggle dataset."
sh(f"cd {WORK} && python {SCRIPTS}/train.py --data {MIX}/data.yaml --model {prior} --epochs {EPOCHS} --lr0 0.002 --imgsz 640 --batch 32 --project {RUNS} --out {WORK}/dartnet.tflite", check=False)

In [ ]:
# 6. Collect the results in /kaggle/working. If the export to TFLite failed, best.pt is still here and can be exported anywhere.
best = f"{RUNS}/dartnet/weights/best.pt"
assert os.path.exists(best), "training did not produce weights: read the training output above"
shutil.copy(best, f"{WORK}/best.pt")
if not os.path.exists(f"{WORK}/dartnet.tflite"):
    found = glob.glob(f"{RUNS}/dartnet/weights/**/*float32.tflite", recursive=True) + glob.glob(f"{RUNS}/dartnet/weights/**/*.tflite", recursive=True)
    if found:
        shutil.copy(found[0], f"{WORK}/dartnet.tflite")
    else:
        from ultralytics import YOLO
        out = YOLO(best).export(format="litert", imgsz=640)
        shutil.copy(str(out), f"{WORK}/dartnet.tflite")
print("dartnet.tflite", round(os.path.getsize(f"{WORK}/dartnet.tflite") / 1e6, 1), "MB")
sh(f"tail -n 4 {RUNS}/dartnet/results.csv", check=False)

In [ ]:
# 8. Tidy up, so the Output tab holds only what you need, and list it.
for leftover in glob.glob(f"{RUNS}/dartnet/weights/*_saved_model") + glob.glob(f"{RUNS}/dartnet/weights/*_float*.tflite"):
    shutil.rmtree(leftover, ignore_errors=True) if os.path.isdir(leftover) else os.remove(leftover)
for name in ("dartnet.tflite", "best.pt"):
    path = f"{WORK}/{name}"
    print(("OK      " if os.path.exists(path) else "missing ") + name)
print("\nDownload from the Output tab of this version (or the file list on the right), copy the .tflite files to app/src/main/assets/ and rebuild the app.")